

<div style="
background:linear-gradient(135deg,#35165F 0%,#512888 100%);
color:white;
padding:30px 34px;
border-radius:14px 14px 6px 6px;
margin:8px 0 24px 0;
border-bottom:7px solid #F3C969;
box-shadow:0 7px 20px rgba(53,22,95,0.16);
font-family:Segoe UI,Arial,sans-serif;
">

<div style="font-size:0.82rem;font-weight:700;letter-spacing:0.10em;text-transform:uppercase;margin-bottom:8px;">
Intro to Data Analytics
</div>
<h1 style="color:white;margin:0 0 8px 0;">Pandas <code style="color:white;background:transparent;">groupby()</code></h1>
<p style="color:white;margin:0;font-size:1.08rem;"><strong>Grouping, Aggregation, Transformation, and Filtering</strong></p>
</div>

<div style="
display:inline-block;
color:#35165F;
background:#F1EBF8;
border:1px solid #D9CBEA;
padding:3px 9px;
border-radius:999px;
font-weight:700;
font-size:0.82rem;
margin-bottom:8px;
font-family:Segoe UI,Arial,sans-serif;
">
Learning Objectives
</div>

By the end of this notebook, you should be able to:

- group rows using one or more columns;
- calculate summary statistics for each group;
- use <code>.agg()</code> for multiple calculations;
- explain the difference between aggregation and transformation;
- use <code>.transform()</code> to create group-based values while preserving rows;
- use <code>.filter()</code> to keep or remove entire groups;
- combine grouping, filtering, sorting, and transformation to answer practical data-analysis questions.


<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:6px solid #512888;
padding:14px 18px;
border-radius:8px;
margin:14px 0;
font-family:Segoe UI,Arial,sans-serif;
">

<b>Core idea:</b> <code>groupby()</code> follows the pattern <b>Split → Apply → Combine</b>.
</div>


In [1]:

import pandas as pd



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">1. Our Example Dataset</h2>
</div>


We will begin with a small sales dataset.

Each row represents one transaction. The dataset includes:

- `city`: where the transaction occurred
- `product`: product sold
- `sales`: dollar amount of the transaction
- `units`: number of units sold

In [2]:

sales = pd.DataFrame({
    "city": [
        "Nacogdoches", "Houston", "Dallas",
        "Nacogdoches", "Houston", "Dallas",
        "Houston", "Dallas", "Nacogdoches", "Houston"
    ],
    "product": [
        "Laptop", "Laptop", "Laptop",
        "Phone", "Phone", "Phone",
        "Tablet", "Tablet", "Tablet", "Laptop"
    ],
    "sales": [
        1200, 1800, 1600,
        900, 1500, 1300,
        700, 800, 650, 2100
    ],
    "units": [
        3, 4, 4,
        5, 8, 7,
        3, 4, 2, 5
    ]
})

sales


,city,product,sales,units
0,Nacogdoches,Laptop,1200,3
1,Houston,Laptop,1800,4
2,Dallas,Laptop,1600,4
3,Nacogdoches,Phone,900,5
4,Houston,Phone,1500,8
5,Dallas,Phone,1300,7
6,Houston,Tablet,700,3
7,Dallas,Tablet,800,4
8,Nacogdoches,Tablet,650,2
9,Houston,Laptop,2100,5



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">2. Basic groupby()</h2>
</div>


Suppose we want to answer:

> What is the total sales amount for each city?

The general syntax is:

```python
df.groupby("grouping_column")["numeric_column"].calculation()
```

For this example:

In [3]:

sales.groupby("city")["sales"].sum()


city
Dallas         3700
Houston        6100
Nacogdoches    2750
Name: sales, dtype: int64


Notice that:

- Pandas grouped rows having the same `city`.
- It then added the `sales` values inside each group.
- The grouping column became the index of the result.

`groupby()` by itself does not usually give us a useful final answer:


In [4]:

sales.groupby("city")



The object above is a **GroupBy object**. We still need to tell Pandas what calculation to perform.

---

## 3. Common Aggregation Functions

An **aggregation** summarizes many values into a smaller number of values.

Common group calculations include:

- `sum()`
- `mean()`
- `min()`
- `max()`
- `count()`
- `median()`
- `std()`


In [5]:

# Average sales in each city
sales.groupby("city")["sales"].mean()


city
Dallas         1233.333333
Houston        1525.000000
Nacogdoches     916.666667
Name: sales, dtype: float64

In [6]:

# Largest transaction in each city
sales.groupby("city")["sales"].max()


city
Dallas         1600
Houston        2100
Nacogdoches    1200
Name: sales, dtype: int64

In [7]:

# Total number of units sold in each city
sales.groupby("city")["units"].sum()


city
Dallas         15
Houston        20
Nacogdoches    10
Name: units, dtype: int64


<div style="
background:#FFF7E3;
border-left:6px solid #F3C969;
padding:16px 18px;
border-radius:8px;
margin:18px 0 10px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<b>🧠 Knowledge Check 1</b><br><br>Using the `sales` DataFrame, determine:

1. the average number of units sold per transaction in each city,
2. the smallest sales transaction in each city,
3. the number of sales records for each city.

Write your code below.</div>

In [8]:

# Your code here



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">4. Keeping the Grouping Column as a Regular Column</h2>
</div>


By default, the grouping column becomes the index.

We can use:

```python
as_index=False
```

to keep it as a regular DataFrame column.

In [9]:

sales.groupby("city", as_index=False)["sales"].sum()


,city,sales
0,Dallas,3700
1,Houston,6100
2,Nacogdoches,2750



Another approach is to use `.reset_index()`:


In [10]:

city_sales = sales.groupby("city")["sales"].sum()

city_sales.reset_index()


,city,sales
0,Dallas,3700
1,Houston,6100
2,Nacogdoches,2750



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">5. Grouping Multiple Numeric Columns</h2>
</div>


We can summarize more than one numerical column at once.

In [11]:

sales.groupby("city")[["sales", "units"]].sum()


,sales,units
city,,
Dallas,3700,15
Houston,6100,20
Nacogdoches,2750,10



This calculates:

- total sales by city,
- total units by city.

---

## 6. Using `.agg()`

The `.agg()` method allows us to calculate several statistics at once.


In [12]:

sales.groupby("city")["sales"].agg(["sum", "mean", "min", "max"])


,sum,mean,min,max
city,,,,
Dallas,3700,1233.333333,800,1600
Houston,6100,1525.000000,700,2100
Nacogdoches,2750,916.666667,650,1200



We can also perform different calculations on different columns:


In [13]:

sales.groupby("city").agg({
    "sales": ["sum", "mean"],
    "units": ["sum", "max"]
})


sales              units    
              sum         mean   sum max
city                                    
Dallas       3700  1233.333333    15   7
Houston      6100  1525.000000    20   8
Nacogdoches  2750   916.666667    10   5


<div style="
background:#FFF7E3;
border-left:6px solid #F3C969;
padding:16px 18px;
border-radius:8px;
margin:18px 0 10px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<b>🧠 Knowledge Check 2</b><br><br>Create a summary table for each product containing:

- total sales,
- average sales,
- total units,
- maximum units in a single transaction.</div>

In [14]:

# Your code here



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">7. Grouping by More Than One Column</h2>
</div>


We can form groups using multiple categorical columns.

Suppose we want to answer:

> What are the total sales for every city-product combination?

In [15]:

sales.groupby(["city", "product"])["sales"].sum()


city         product
Dallas       Laptop     1600
             Phone      1300
             Tablet      800
Houston      Laptop     3900
             Phone      1500
             Tablet      700
Nacogdoches  Laptop     1200
             Phone       900
             Tablet      650
Name: sales, dtype: int64


For a cleaner DataFrame:


In [16]:

sales.groupby(
    ["city", "product"],
    as_index=False
)["sales"].sum()


,city,product,sales
0,Dallas,Laptop,1600
1,Dallas,Phone,1300
2,Dallas,Tablet,800
3,Houston,Laptop,3900
4,Houston,Phone,1500
5,Houston,Tablet,700
6,Nacogdoches,Laptop,1200
7,Nacogdoches,Phone,900
8,Nacogdoches,Tablet,650



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">8. Counting Groups: size() vs. count()</h2>
</div>


To count the number of rows in each group:

In [17]:

sales.groupby("city").size()


city
Dallas         3
Houston        4
Nacogdoches    3
dtype: int64


`size()` counts rows.

`count()` counts **non-missing values** in a particular column:


In [18]:

sales.groupby("city")["sales"].count()


city
Dallas         3
Houston        4
Nacogdoches    3
Name: sales, dtype: int64


<div style="
background:#F1EBF8;
border-left:6px solid #512888;
padding:14px 18px;
border-radius:8px;
margin:14px 0;
font-family:Segoe UI,Arial,sans-serif;
">
These may give different answers if the dataset contains missing values.

---

# Part II — Group-Based Transformation

## 9. What Is a Transformation?

Aggregation and transformation are related, but they behave differently.

### Aggregation

Aggregation reduces each group to a smaller result.

For example:</div>

In [19]:

sales.groupby("city")["sales"].mean()


city
Dallas         1233.333333
Houston        1525.000000
Nacogdoches     916.666667
Name: sales, dtype: float64


If there are three cities, the result contains only three values.

### Transformation

A transformation calculates something for each group **but returns a value for every original row**.

This is useful when we want to compare each row to some group-level statistic.

The general syntax is:

```python
df.groupby("column")["value"].transform("function")
```



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">10. Add the Average Sales of Each City to Every Row</h2>
</div>


Suppose we want each transaction to also show the average sales amount for its city.

In [20]:

sales["city_avg_sales"] = (
    sales.groupby("city")["sales"]
         .transform("mean")
)

sales


,city,product,sales,units,city_avg_sales
0,Nacogdoches,Laptop,1200,3,916.666667
1,Houston,Laptop,1800,4,1525.000000
2,Dallas,Laptop,1600,4,1233.333333
3,Nacogdoches,Phone,900,5,916.666667
4,Houston,Phone,1500,8,1525.000000
5,Dallas,Phone,1300,7,1233.333333
6,Houston,Tablet,700,3,1525.000000
7,Dallas,Tablet,800,4,1233.333333
8,Nacogdoches,Tablet,650,2,916.666667
9,Houston,Laptop,2100,5,1525.000000



Observe what happened:

- every Houston row receives Houston's average,
- every Dallas row receives Dallas's average,
- every Nacogdoches row receives Nacogdoches's average.

The number of rows remains unchanged.

This is the key difference between:

```python
groupby(...).mean()
```

and:

```python
groupby(...).transform("mean")
```



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">11. Compare Each Transaction to Its City's Average</h2>
</div>


We can now calculate how far each transaction is above or below the city average.

In [21]:

sales["difference_from_city_avg"] = (
    sales["sales"] - sales["city_avg_sales"]
)

sales


,city,product,sales,units,city_avg_sales,difference_from_city_avg
0,Nacogdoches,Laptop,1200,3,916.666667,283.333333
1,Houston,Laptop,1800,4,1525.000000,275.000000
2,Dallas,Laptop,1600,4,1233.333333,366.666667
3,Nacogdoches,Phone,900,5,916.666667,-16.666667
4,Houston,Phone,1500,8,1525.000000,-25.000000
5,Dallas,Phone,1300,7,1233.333333,66.666667
6,Houston,Tablet,700,3,1525.000000,-825.000000
7,Dallas,Tablet,800,4,1233.333333,-433.333333
8,Nacogdoches,Tablet,650,2,916.666667,-266.666667
9,Houston,Laptop,2100,5,1525.000000,575.000000



A positive value means that the transaction is above the city's average.

A negative value means that the transaction is below the city's average.



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">12. Create a Group-Based Percentage</h2>
</div>


We can also determine what percentage of a city's total sales came from each transaction.

First calculate the city total for every row:

In [22]:

sales["city_total_sales"] = (
    sales.groupby("city")["sales"]
         .transform("sum")
)

sales


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales
0,Nacogdoches,Laptop,1200,3,916.666667,283.333333,2750
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700
3,Nacogdoches,Phone,900,5,916.666667,-16.666667,2750
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700
6,Houston,Tablet,700,3,1525.000000,-825.000000,6100
7,Dallas,Tablet,800,4,1233.333333,-433.333333,3700
8,Nacogdoches,Tablet,650,2,916.666667,-266.666667,2750
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100



Now divide each transaction by its city's total:


In [23]:

sales["percent_of_city_sales"] = (
    sales["sales"] / sales["city_total_sales"] * 100
)

sales


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales,percent_of_city_sales
0,Nacogdoches,Laptop,1200,3,916.666667,283.333333,2750,43.636364
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100,29.508197
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700,43.243243
3,Nacogdoches,Phone,900,5,916.666667,-16.666667,2750,32.727273
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100,24.590164
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700,35.135135
6,Houston,Tablet,700,3,1525.000000,-825.000000,6100,11.475410
7,Dallas,Tablet,800,4,1233.333333,-433.333333,3700,21.621622
8,Nacogdoches,Tablet,650,2,916.666667,-266.666667,2750,23.636364
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100,34.426230



This is a very useful real-world application of `transform()`.

For each row, we now know:

> What percentage of the city's total sales came from this transaction?



<div style="
background:#FFF7E3;
border-left:6px solid #F3C969;
padding:16px 18px;
border-radius:8px;
margin:18px 0 10px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<b>🧠 Knowledge Check 3 — Transformation</b><br><br>Using `transform()`:

1. Add a column containing the average number of units sold for each product.
2. Add another column showing the difference between each transaction's units and its product's average units.
3. Add a column containing the total sales for each product.

Do not reduce the number of rows.</div>

In [24]:

# Your code here



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">Part III — Filtering Groups [Optional]</h2>
</div>


## 13. Row Filtering vs. Group Filtering

You already know how to filter individual rows.

For example:

In [25]:

sales[sales["sales"] > 1000]


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales,percent_of_city_sales
0,Nacogdoches,Laptop,1200,3,916.666667,283.333333,2750,43.636364
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100,29.508197
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700,43.243243
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100,24.590164
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700,35.135135
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100,34.426230



This asks:

> Which individual transactions have sales greater than \$1000?

But sometimes we want to keep or remove an **entire group** depending on a group-level condition.

For that we can use:

```python
groupby(...).filter(...)
```



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">14. Keep Cities Whose Total Sales Exceed \$3000</h2>
</div>


We want to:

1. create groups by city,
2. calculate the total sales of each group,
3. keep every row belonging to cities whose total exceeds \$3000.

In [26]:

sales.groupby("city").filter(
    lambda group: group["sales"].sum() > 3000
)


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales,percent_of_city_sales
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100,29.508197
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700,43.243243
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100,24.590164
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700,35.135135
6,Houston,Tablet,700,3,1525.000000,-825.000000,6100,11.475410
7,Dallas,Tablet,800,4,1233.333333,-433.333333,3700,21.621622
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100,34.426230



The `group` inside the lambda represents one city's subset of the DataFrame.

For each city, Pandas checks:

```python
group["sales"].sum() > 3000
```

If the condition is `True`, the **entire city's rows are kept**.

If it is `False`, the entire group is removed.



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">15. Filter Groups Based on Average Sales</h2>
</div>


Keep cities whose average transaction is greater than \$1200:

In [27]:

sales.groupby("city").filter(
    lambda group: group["sales"].mean() > 1200
)


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales,percent_of_city_sales
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100,29.508197
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700,43.243243
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100,24.590164
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700,35.135135
6,Houston,Tablet,700,3,1525.000000,-825.000000,6100,11.475410
7,Dallas,Tablet,800,4,1233.333333,-433.333333,3700,21.621622
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100,34.426230



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">16. Filter Groups by Group Size</h2>
</div>


Suppose we want to keep only cities that have at least 3 transaction records:

In [28]:

sales.groupby("city").filter(
    lambda group: len(group) >= 3
)


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales,percent_of_city_sales
0,Nacogdoches,Laptop,1200,3,916.666667,283.333333,2750,43.636364
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100,29.508197
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700,43.243243
3,Nacogdoches,Phone,900,5,916.666667,-16.666667,2750,32.727273
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100,24.590164
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700,35.135135
6,Houston,Tablet,700,3,1525.000000,-825.000000,6100,11.475410
7,Dallas,Tablet,800,4,1233.333333,-433.333333,3700,21.621622
8,Nacogdoches,Tablet,650,2,916.666667,-266.666667,2750,23.636364
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100,34.426230



We can also use:

```python
group.shape[0]
```

to determine the number of rows.


In [29]:

sales.groupby("city").filter(
    lambda group: group.shape[0] >= 3
)


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales,percent_of_city_sales
0,Nacogdoches,Laptop,1200,3,916.666667,283.333333,2750,43.636364
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100,29.508197
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700,43.243243
3,Nacogdoches,Phone,900,5,916.666667,-16.666667,2750,32.727273
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100,24.590164
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700,35.135135
6,Houston,Tablet,700,3,1525.000000,-825.000000,6100,11.475410
7,Dallas,Tablet,800,4,1233.333333,-433.333333,3700,21.621622
8,Nacogdoches,Tablet,650,2,916.666667,-266.666667,2750,23.636364
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100,34.426230



<div style="
background:#FFF7E3;
border-left:6px solid #F3C969;
padding:16px 18px;
border-radius:8px;
margin:18px 0 10px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<b>🧠 Knowledge Check 4 — Group Filtering</b><br><br>Use `groupby().filter()` to answer the following:

1. Keep products whose total sales are greater than \$2500.
2. Keep cities whose average units per transaction are at least 4.
3. Keep products that appear in at least 3 transactions.</div>

In [30]:

# Your code here



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">Part IV — Combining Filtering and Grouping [Optional]</h2>
</div>


## 17. Filter Rows First, Then Group

Suppose we are interested only in transactions greater than \$1000.

First filter the rows:

In [31]:

high_sales = sales[sales["sales"] > 1000]

high_sales


,city,product,sales,units,city_avg_sales,difference_from_city_avg,city_total_sales,percent_of_city_sales
0,Nacogdoches,Laptop,1200,3,916.666667,283.333333,2750,43.636364
1,Houston,Laptop,1800,4,1525.000000,275.000000,6100,29.508197
2,Dallas,Laptop,1600,4,1233.333333,366.666667,3700,43.243243
4,Houston,Phone,1500,8,1525.000000,-25.000000,6100,24.590164
5,Dallas,Phone,1300,7,1233.333333,66.666667,3700,35.135135
9,Houston,Laptop,2100,5,1525.000000,575.000000,6100,34.426230



Then group the remaining rows:


In [32]:

high_sales.groupby("city")["sales"].sum()


city
Dallas         2900
Houston        5400
Nacogdoches    1200
Name: sales, dtype: int64


This answers:

> Among transactions above \$1000, what is the total sales amount for each city?



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">18. Group First, Then Filter the Summary</h2>
</div>


We can also first calculate a summary:

In [33]:

city_totals = sales.groupby("city")["sales"].sum()

city_totals


city
Dallas         3700
Houston        6100
Nacogdoches    2750
Name: sales, dtype: int64


Then filter that summary:


In [34]:

city_totals[city_totals > 3000]


city
Dallas     3700
Houston    6100
Name: sales, dtype: int64


These are three different ideas:

### Row filtering

```python
sales[sales["sales"] > 1000]
```

Keeps individual rows.

### Filtering an aggregated result

```python
city_totals[city_totals > 3000]
```

Keeps summary results.

### Group filtering

```python
sales.groupby("city").filter(
    lambda group: group["sales"].sum() > 3000
)
```

Keeps all original rows belonging to qualifying groups.



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">Part V — Sorting Grouped Results</h2>
</div>


## 19. Sort Groups by Their Summary Statistic

A very common analysis pattern is:

```python
groupby → aggregate → sort
```

For example:

In [35]:

sales.groupby("city")["sales"] \
     .sum() \
     .sort_values(ascending=False)


city
Houston        6100
Dallas         3700
Nacogdoches    2750
Name: sales, dtype: int64


This answers:

> Which cities generated the most total sales?

Similarly:


In [36]:

sales.groupby("product")["units"] \
     .sum() \
     .sort_values(ascending=False)


product
Phone     20
Laptop    16
Tablet     9
Name: units, dtype: int64


<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">20. Find the Group With the Largest Value</h2>
</div>


We can use `.idxmax()`.

In [37]:

product_sales = sales.groupby("product")["sales"].sum()

product_sales


product
Laptop    6700
Phone     3700
Tablet    2150
Name: sales, dtype: int64

In [38]:

product_sales.idxmax()


'Laptop'


The result is the product associated with the largest total sales value.



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">Part VI — A More Realistic Dataset</h2>
</div>


The following dataset contains university employee information.

In [39]:

employees = pd.DataFrame({
    "department": [
        "Computer Science", "Computer Science", "Mathematics",
        "Mathematics", "Computer Science", "Biology",
        "Biology", "Mathematics", "Biology", "Computer Science"
    ],
    "rank": [
        "Professor", "Associate", "Professor",
        "Assistant", "Assistant", "Professor",
        "Associate", "Associate", "Assistant", "Professor"
    ],
    "salary": [
        95000, 78000, 92000,
        68000, 65000, 90000,
        76000, 80000, 64000, 101000
    ],
    "years": [
        18, 10, 20,
        4, 3, 17,
        9, 11, 2, 22
    ]
})

employees


,department,rank,salary,years
0,Computer Science,Professor,95000,18
1,Computer Science,Associate,78000,10
2,Mathematics,Professor,92000,20
3,Mathematics,Assistant,68000,4
4,Computer Science,Assistant,65000,3
5,Biology,Professor,90000,17
6,Biology,Associate,76000,9
7,Mathematics,Associate,80000,11
8,Biology,Assistant,64000,2
9,Computer Science,Professor,101000,22



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">21. Average Salary by Department</h2>
</div>


In [40]:

employees.groupby("department")["salary"].mean()


department
Biology             76666.666667
Computer Science    84750.000000
Mathematics         80000.000000
Name: salary, dtype: float64


<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">22. Salary Summary by Department</h2>
</div>


In [41]:

employees.groupby("department")["salary"].agg(
    ["min", "max", "mean", "median"]
)


,min,max,mean,median
department,,,,
Biology,64000,90000,76666.666667,76000.0
Computer Science,65000,101000,84750.000000,86500.0
Mathematics,68000,92000,80000.000000,80000.0



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">23. Average Salary by Department and Rank</h2>
</div>


In [42]:

employees.groupby(
    ["department", "rank"],
    as_index=False
)["salary"].mean()


,department,rank,salary
0,Biology,Assistant,64000.0
1,Biology,Associate,76000.0
2,Biology,Professor,90000.0
3,Computer Science,Assistant,65000.0
4,Computer Science,Associate,78000.0
5,Computer Science,Professor,98000.0
6,Mathematics,Assistant,68000.0
7,Mathematics,Associate,80000.0
8,Mathematics,Professor,92000.0



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">24. Transformation Example: Department Average Salary</h2>
</div>


Suppose we want to compare every employee's salary with the average salary of their department.

In [43]:

employees["department_avg_salary"] = (
    employees.groupby("department")["salary"]
             .transform("mean")
)

employees


,department,rank,salary,years,department_avg_salary
0,Computer Science,Professor,95000,18,84750.000000
1,Computer Science,Associate,78000,10,84750.000000
2,Mathematics,Professor,92000,20,80000.000000
3,Mathematics,Assistant,68000,4,80000.000000
4,Computer Science,Assistant,65000,3,84750.000000
5,Biology,Professor,90000,17,76666.666667
6,Biology,Associate,76000,9,76666.666667
7,Mathematics,Associate,80000,11,80000.000000
8,Biology,Assistant,64000,2,76666.666667
9,Computer Science,Professor,101000,22,84750.000000


In [44]:

employees["difference_from_department_avg"] = (
    employees["salary"] -
    employees["department_avg_salary"]
)

employees


,department,rank,salary,years,department_avg_salary,difference_from_department_avg
0,Computer Science,Professor,95000,18,84750.000000,10250.000000
1,Computer Science,Associate,78000,10,84750.000000,-6750.000000
2,Mathematics,Professor,92000,20,80000.000000,12000.000000
3,Mathematics,Assistant,68000,4,80000.000000,-12000.000000
4,Computer Science,Assistant,65000,3,84750.000000,-19750.000000
5,Biology,Professor,90000,17,76666.666667,13333.333333
6,Biology,Associate,76000,9,76666.666667,-666.666667
7,Mathematics,Associate,80000,11,80000.000000,0.000000
8,Biology,Assistant,64000,2,76666.666667,-12666.666667
9,Computer Science,Professor,101000,22,84750.000000,16250.000000



Now every employee remains in the DataFrame, but each row contains additional group-level information.

This type of transformation is very useful when answering questions such as:

- Is this observation above or below its group's average?
- What percentage of the group total does this observation represent?
- How does each employee compare with their department?
- How does each transaction compare with its product category?



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">25. Filtering Example: Keep Departments With High Average Salary</h2>
</div>


Keep all employees belonging to departments whose average salary exceeds \$80,000.

In [45]:

employees.groupby("department").filter(
    lambda group: group["salary"].mean() > 80000
)


,department,rank,salary,years,department_avg_salary,difference_from_department_avg
0,Computer Science,Professor,95000,18,84750.0,10250.0
1,Computer Science,Associate,78000,10,84750.0,-6750.0
4,Computer Science,Assistant,65000,3,84750.0,-19750.0
9,Computer Science,Professor,101000,22,84750.0,16250.0



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">26. Filtering Example: Keep Departments With at Least 3 Employees</h2>
</div>


In [46]:

employees.groupby("department").filter(
    lambda group: len(group) >= 3
)


,department,rank,salary,years,department_avg_salary,difference_from_department_avg
0,Computer Science,Professor,95000,18,84750.000000,10250.000000
1,Computer Science,Associate,78000,10,84750.000000,-6750.000000
2,Mathematics,Professor,92000,20,80000.000000,12000.000000
3,Mathematics,Assistant,68000,4,80000.000000,-12000.000000
4,Computer Science,Assistant,65000,3,84750.000000,-19750.000000
5,Biology,Professor,90000,17,76666.666667,13333.333333
6,Biology,Associate,76000,9,76666.666667,-666.666667
7,Mathematics,Associate,80000,11,80000.000000,0.000000
8,Biology,Assistant,64000,2,76666.666667,-12666.666667
9,Computer Science,Professor,101000,22,84750.000000,16250.000000



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">Part VII — Data Analysis Practice</h2>
</div>


Use the `employees` DataFrame to answer the following questions.

Do not worry about using particular variable names. Focus on selecting the correct Pandas operations.

### Basic Grouping

1. What is the average salary in each department?
2. What is the maximum salary in each department?
3. How many employees belong to each department?
4. What is the average number of years of experience in each department?
5. What is the total salary represented by each department?

### Multiple Groups

6. What is the average salary for each academic rank?
7. What is the average salary for every department-rank combination?
8. How many employees belong to each department-rank combination?

### Aggregation

9. Create a table showing the minimum, maximum, mean, and median salary for each department.
10. Create a table showing total salary and average years of experience for each department.

### Transformation

11. Add a column containing the average years of experience for each employee's department.
12. For each employee, calculate how many years above or below their department's average experience they are.
13. Add a column containing the maximum salary in each employee's department.
14. Calculate each employee's salary as a percentage of the total salary of their department.

### Group Filtering

15. Keep only departments whose average salary is greater than \$75,000.
16. Keep only departments having at least three employees.
17. Keep only ranks whose average salary exceeds \$80,000.
18. Keep only departments where the maximum salary exceeds \$95,000.

### Analysis

19. Which department has the highest average salary?
20. Which academic rank has the highest average salary?
21. Which department has the greatest total salary?
22. Which department has the greatest average years of experience?

In [47]:

# Use this cell for your analysis.



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">Part VIII — Final Challenge</h2>
</div>


Suppose an analyst wants to identify employees who earn more than their department average.

Complete the following analysis:

1. Use `groupby()` and `transform()` to determine the average salary for each employee's department.
2. Compare each employee's salary with that department average.
3. Keep only the employees earning above their department average.
4. Sort those employees from highest salary to lowest salary.

Try to solve this using a combination of:

- `groupby()`
- `transform()`
- Boolean filtering
- `sort_values()`

In [48]:

# Your solution here



<div style="
background:#FBF9FE;
border:1px solid #D9CBEA;
border-left:7px solid #512888;
padding:14px 18px;
border-radius:10px;
margin:22px 0 12px 0;
font-family:Segoe UI,Arial,sans-serif;
">
<h2 style="margin:0;color:#35165F;border:none;">Summary</h2>
</div>


## Aggregation

Aggregation reduces each group to one or more summary values.

```python
df.groupby("group")["value"].mean()
```

---

## Multiple Aggregations

```python
df.groupby("group")["value"].agg(
    ["min", "max", "mean"]
)
```

---

## Grouping by Multiple Columns

```python
df.groupby(["group1", "group2"])["value"].mean()
```

---

## Transformation

Transformation calculates a group-based value while preserving the original number of rows.

```python
df["group_mean"] = (
    df.groupby("group")["value"]
      .transform("mean")
)
```

---

## Group Filtering

`filter()` keeps or removes **entire groups**.

```python
df.groupby("group").filter(
    lambda g: g["value"].mean() > 100
)
```

---

## A Useful Analysis Workflow

```python
df.groupby("category")["value"] \
  .mean() \
  .sort_values(ascending=False)
```

---

### Most Important Distinction

| Operation | What happens to rows? | Example |
|---|---|---|
| Aggregation | Number of rows decreases | `groupby().mean()` |
| Transformation | Original rows remain | `groupby().transform("mean")` |
| Group filtering | Entire groups are kept or removed | `groupby().filter(...)` |

Understanding this distinction will make `groupby()` much easier to use in real data analysis.